In [ ]:
from pathlib import Path

# Local Jupyter does not have /content. Colab does. Write a scratch secrets file
# only when the target parent directory exists. Fill HF_TOKEN yourself in Colab
# if the embedding model download requires it.
secrets_target = Path("/content/secrets.env")
if secrets_target.parent.exists():
    if not secrets_target.exists():
        secrets_target.write_text(
            """# Optional Colab-only secrets scratch file.
# HF_TOKEN="paste-your-token-here-if-needed"
""",
            encoding="utf-8",
        )
    print(f"Using secrets file path: {secrets_target}")
else:
    print("Skipping /content/secrets.env bootstrap because this runtime is not Colab.")


In [ ]:
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IS_COLAB_LIKE = Path("/content").exists()


def _assert_python_version(min_major: int = 3, min_minor: int = 10) -> None:
    major, minor = sys.version_info.major, sys.version_info.minor
    if (major, minor) < (min_major, min_minor):
        raise RuntimeError(
            f"Python {major}.{minor} is too old — project requires >= {min_major}.{min_minor}."
        )
    print(f"Python {platform.python_version()} OK")


def _assert_gpu_available() -> None:
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        location_hint = (
            "This looks like a local Jupyter runtime without NVIDIA tooling. "
            if not IS_COLAB_LIKE else ""
        )
        raise RuntimeError(
            location_hint +
            "nvidia-smi not available. Run this notebook in a GPU-backed runtime "
            "(for example Colab with a GPU attached) before starting the export."
        )
    output = subprocess.check_output(
        [nvidia_smi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
        text=True,
        timeout=10,
    )
    gpu_line = output.strip().splitlines()[0] if output.strip() else ""
    if not gpu_line:
        raise RuntimeError("GPU present but no device info returned; retry the runtime.")
    print(f"GPU: {gpu_line}")


def _assert_basic_tools() -> None:
    required_tools = ("git", "curl") if not IS_COLAB_LIKE else ("curl", "apt-get", "git")
    for tool in required_tools:
        if shutil.which(tool) is None:
            raise RuntimeError(f"Required tool '{tool}' is missing from PATH.")
    print(f"Shell tools OK: {', '.join(required_tools)}")


_assert_python_version()
_assert_gpu_available()
_assert_basic_tools()
print("\nRuntime sanity: PASS")


In [ ]:
from pathlib import Path

DRIVE_MOUNT = Path("/content/drive")
DRIVE_READY = False

try:
    from google.colab import drive  # type: ignore[import-not-found]
except ImportError:
    print("google.colab not available; skipping Drive mount (non-Colab runtime).")
else:
    my_drive = DRIVE_MOUNT / "MyDrive"
    if my_drive.is_dir():
        print(f"Drive already mounted at {DRIVE_MOUNT}")
        DRIVE_READY = True
    else:
        drive.mount(str(DRIVE_MOUNT))  # type: ignore[attr-defined]
        DRIVE_READY = my_drive.is_dir()

if DRIVE_READY:
    SYNTHEA_DIR = DRIVE_MOUNT / "MyDrive/kubuntu/agentic-memory/big-healtcare-data/synthetic-data"
    EXPERIMENT_DIR = DRIVE_MOUNT / "MyDrive/kubuntu/agentic-memory/experiments/healthcare"
else:
    SYNTHEA_DIR = Path("/content/synthea")
    EXPERIMENT_DIR = Path("/content/experiments/healthcare")
    SYNTHEA_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_DIR = EXPERIMENT_DIR / "checkpoints"
RESULTS_DIR = EXPERIMENT_DIR / "results"
EXPORT_DIR = EXPERIMENT_DIR / "embedded-exports"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for label, path in [
    ("Drive mount", DRIVE_MOUNT if DRIVE_READY else Path("/content (local)")),
    ("Synthea source", SYNTHEA_DIR),
    ("Experiment dir", EXPERIMENT_DIR),
    ("Checkpoints", CHECKPOINT_DIR),
    ("Results", RESULTS_DIR),
    ("Embedded exports", EXPORT_DIR),
]:
    marker = "OK" if path.exists() else "MISSING"
    print(f"[{marker}] {label}: {path}")


In [ ]:
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

REPO_URL = "https://github.com/jarmen423/agentic-memory.git"
REPO_REF = "codex/export-embedded-healthcare"
COLAB_REPO = Path("/content/agentic-memory")
PREFER_REMOTE_REPO_ON_COLAB = True


def _find_repo_root() -> Path | None:
    if COLAB_REPO.is_dir() and (COLAB_REPO / "pyproject.toml").is_file():
        return COLAB_REPO.resolve()
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "pyproject.toml").is_file() and (base / "src" / "agentic_memory").is_dir():
            return base
    return None


def _is_git_repo(path: Path) -> bool:
    return path.is_dir() and (path / ".git").exists()


def _looks_like_uploaded_tree(path: Path) -> bool:
    return path.is_dir() and (path / "src" / "agentic_memory" / "__init__.py").exists() and not (path / ".git").exists()


def _backup_uploaded_tree(path: Path) -> None:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
    backup = path.parent / f"{path.name}-uploaded-backup-{stamp}"
    shutil.move(str(path), str(backup))
    print(f"Moved uploaded tree aside to {backup}")


def _ensure_remote_colab_repo() -> Path:
    if _looks_like_uploaded_tree(COLAB_REPO):
        _backup_uploaded_tree(COLAB_REPO)

    if _is_git_repo(COLAB_REPO):
        print(f"Repo already present at {COLAB_REPO}; force-syncing to origin/{REPO_REF}")
        subprocess.check_call(["git", "-C", str(COLAB_REPO), "fetch", "origin", REPO_REF, "--tags"])
        subprocess.check_call(["git", "-C", str(COLAB_REPO), "checkout", REPO_REF])
        subprocess.check_call(["git", "-C", str(COLAB_REPO), "reset", "--hard", f"origin/{REPO_REF}"])
        subprocess.check_call(["git", "-C", str(COLAB_REPO), "clean", "-fd"])
        return COLAB_REPO

    if COLAB_REPO.exists():
        raise RuntimeError(
            f"{COLAB_REPO} exists but is neither a git checkout nor a recognised uploaded tree. "
            "Move it aside manually, then re-run this cell."
        )

    print(f"Cloning {REPO_URL}@{REPO_REF} into {COLAB_REPO}")
    subprocess.check_call(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(COLAB_REPO)])
    return COLAB_REPO


if Path("/content").is_dir() and PREFER_REMOTE_REPO_ON_COLAB:
    REPO_DIR = _ensure_remote_colab_repo().resolve()
else:
    _found = _find_repo_root()
    if _found is None:
        raise RuntimeError("Could not find the agentic-memory repository.")
    REPO_DIR = _found

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"], text=True).strip()
print(f"Repo at commit {commit}")
os.environ["PYTHONPATH"] = f"{REPO_DIR}/src:{os.environ.get('PYTHONPATH', '')}"
os.chdir(REPO_DIR)


In [ ]:
import importlib
import subprocess
import sys
from pathlib import Path

COLAB_REPO = Path("/content/agentic-memory")


def _repo_root() -> Path:
    if COLAB_REPO.is_dir() and (COLAB_REPO / "pyproject.toml").is_file():
        return COLAB_REPO.resolve()
    here = Path.cwd().resolve()
    for base in [here, *here.parents]:
        if (base / "pyproject.toml").is_file() and (base / "src" / "agentic_memory").is_dir():
            return base
    raise RuntimeError("Could not find repo root.")


REPO_DIR = _repo_root()
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "transformers>=4.44",
    "accelerate>=0.33",
    "sentencepiece>=0.2",
    "einops>=0.8",
])

repo_src = REPO_DIR / "src"
if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))
importlib.invalidate_caches()
importlib.import_module("agentic_memory")
print(f"[ok] agentic_memory is importable (repo={REPO_DIR})")


In [ ]:
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Literal
import json
import os

Tier = Literal["smoke", "mid", "scale", "full"]
TIER: Tier = "mid"
SKIP_OBSERVATIONS = False


@dataclass
class RunConfig:
    tier: Tier
    max_patients: int
    checkpoint_every: int
    embed_batch_size: int
    neo4j_write_batch_size: int
    embedding_dim: int
    source_tarball: Path
    project_id: str
    checkpoint_path: Path
    results_path: Path
    export_dir: Path
    export_manifest_path: Path


_TIER_PRESETS: dict[Tier, dict] = {
    "smoke": {"max_patients": 2_000, "checkpoint_every": 250, "embed_batch_size": 64, "neo4j_write_batch_size": 500, "source_tarball_name": "synthea_2017_02_27.tar.gz"},
    "mid": {"max_patients": 20_000, "checkpoint_every": 1_000, "embed_batch_size": 128, "neo4j_write_batch_size": 1_000, "source_tarball_name": "synthea_2017_02_27.tar.gz"},
    "scale": {"max_patients": 200_000, "checkpoint_every": 2_000, "embed_batch_size": 128, "neo4j_write_batch_size": 1_000, "source_tarball_name": "synthea_1m_fhir_3_0_May_24.tar.gz"},
    "full": {"max_patients": 1_000_000, "checkpoint_every": 5_000, "embed_batch_size": 256, "neo4j_write_batch_size": 2_000, "source_tarball_name": "synthea_1m_fhir_3_0_May_24.tar.gz"},
}


def build_run_config(tier: Tier) -> RunConfig:
    preset = _TIER_PRESETS[tier]
    return RunConfig(
        tier=tier,
        max_patients=preset["max_patients"],
        checkpoint_every=preset["checkpoint_every"],
        embed_batch_size=preset["embed_batch_size"],
        neo4j_write_batch_size=preset["neo4j_write_batch_size"],
        embedding_dim=2048,
        source_tarball=SYNTHEA_DIR / preset["source_tarball_name"],
        project_id=f"synthea-scale-{tier}",
        checkpoint_path=CHECKPOINT_DIR / f"{tier}.json",
        results_path=RESULTS_DIR / f"export_{tier}.json",
        export_dir=EXPORT_DIR / tier,
        export_manifest_path=EXPORT_DIR / tier / "manifest.json",
    )


CONFIG = build_run_config(TIER)
os.environ["EMBEDDING_PROVIDER"] = "nemotron_local"
print(json.dumps({k: str(v) for k, v in asdict(CONFIG).items()}, indent=2, default=str))


In [ ]:
from pathlib import Path

CONFIG.source_tarball = Path(
    "/content/drive/MyDrive/kubuntu/agentic-memory/big-healtcare-data/synthetic-data/synthea_1m_fhir_3_0_May_24.tar.gz"
)
if not CONFIG.source_tarball.exists():
    raise RuntimeError(f"Drive tarball not found: {CONFIG.source_tarball}")

CONFIG.project_id = "synthea-scale-mid-r3-drive"
CONFIG.export_dir = EXPORT_DIR / "mid-r3-drive"
CONFIG.export_manifest_path = CONFIG.export_dir / "manifest.json"
print("Source path:", CONFIG.source_tarball)
print("Export dir:", CONFIG.export_dir)


In [ ]:
import time
from agentic_memory.healthcare.fhir_loader import SyntheaFHIRLoader

_loader = SyntheaFHIRLoader(str(CONFIG.source_tarball), max_patients=1)
start = time.monotonic()
first_record = next(iter(_loader.iter_records()), None)
elapsed = time.monotonic() - start

if first_record is None:
    raise RuntimeError(f"No healthcare records found in {CONFIG.source_tarball}")

print(f"First record yielded in {elapsed:.1f}s")
print(f"record_type = {first_record.get('record_type', '<missing>')}")


In [ ]:
import os
from pathlib import Path


def _ensure_hf_token_from_dotenv_files() -> None:
    if os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN"):
        return
    paths = (Path("/content/secrets.env"), Path("/content/.env"), Path("/content/agentic-memory/experiments/healthcare/.env"))
    for path in paths:
        if not path.is_file():
            continue
        for raw in path.read_text().splitlines():
            line = raw.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, val = line.split("=", 1)
            key, val = key.strip(), val.strip().strip('"').strip("'")
            if key not in ("HF_TOKEN", "HUGGING_FACE_HUB_TOKEN") or not val:
                continue
            os.environ.setdefault("HF_TOKEN", val)
            os.environ.setdefault("HUGGING_FACE_HUB_TOKEN", val)
            print(f"[embed] Set HF_TOKEN from {path}")
            return


_ensure_hf_token_from_dotenv_files()
from agentic_memory.core.embedding import EmbeddingService

embedder = EmbeddingService(
    provider="nemotron_local",
    model="nvidia/llama-nemotron-embed-vl-1b-v2",
    api_key=os.environ.get("NVIDIA_API_KEY"),
)
print(f"[embed] Resolved local model: {embedder.model}")
_warm_vectors = embedder.embed_batch([
    "Condition: Hypertensive disorder, systemic arterial (disorder)",
    "Medication: Lisinopril 10 MG Oral Tablet",
    "Observation: Systolic blood pressure 142 mmHg",
])
detected_dim = len(_warm_vectors[0])
if CONFIG.embedding_dim != detected_dim:
    print(f"Embedding dim mismatch: CONFIG was {CONFIG.embedding_dim}, model produced {detected_dim}. Updating CONFIG to {detected_dim}.")
    CONFIG.embedding_dim = detected_dim
print(f"Nemotron local-GPU embedding service ready. dim={detected_dim}")


In [ ]:
print("resolved model:", embedder.model)
print("dimensions:", embedder.dimensions)
import os
print("AM_LOCAL_EMBED_MODEL:", os.environ.get("AM_LOCAL_EMBED_MODEL"))


## Section 6 — Export embedded chunks for the VM

This notebook now ends at the embedding-export stage. It does **not** connect to Neo4j.
Run it to generate `chunk-*.jsonl.gz` files plus `manifest.json`, then shut down the GPU runtime
and import those chunks on the Hetzner VM.


In [ ]:
from pathlib import Path
import shlex
import sys

EXPORT_SCRIPT = REPO_DIR / "scripts" / "export_embedded_synthea.py"
if not EXPORT_SCRIPT.exists():
    raise RuntimeError(f"Export script missing: {EXPORT_SCRIPT}")

CONFIG.export_dir.mkdir(parents=True, exist_ok=True)
export_command = [
    sys.executable,
    str(EXPORT_SCRIPT),
    "--data-dir", str(CONFIG.source_tarball),
    "--output-dir", str(CONFIG.export_dir),
    "--project-id", CONFIG.project_id,
    "--max-patients", str(CONFIG.max_patients),
    "--chunk-records", str(CONFIG.neo4j_write_batch_size * 8),
    "--embed-batch-size", str(CONFIG.embed_batch_size),
    "--overwrite",
]
print("Export destination:", CONFIG.export_dir)
print("Manifest path:", CONFIG.export_manifest_path)
print("Command preview:")
print(" ".join(shlex.quote(part) for part in export_command))


In [ ]:
import os
import subprocess
import time

os.environ["EMBEDDING_PROVIDER"] = "nemotron_local"
export_start = time.monotonic()
subprocess.run(export_command, check=True, cwd=str(REPO_DIR))
export_elapsed_min = (time.monotonic() - export_start) / 60
print(f"Embedded export complete in {export_elapsed_min:.1f} minutes")


In [ ]:
import json

if not CONFIG.export_manifest_path.exists():
    raise RuntimeError(f"Export finished but manifest is missing: {CONFIG.export_manifest_path}")

manifest = json.loads(CONFIG.export_manifest_path.read_text(encoding="utf-8"))
chunk_paths = sorted(CONFIG.export_dir.glob("chunk-*.jsonl.gz"))
summary = {
    "project_id": manifest.get("project_id"),
    "embedding_provider": manifest.get("embedding_provider"),
    "embedding_model": manifest.get("embedding_model"),
    "total_rows": manifest.get("total_rows"),
    "chunk_count": len(chunk_paths),
    "output_dir": str(CONFIG.export_dir),
    "manifest_path": str(CONFIG.export_manifest_path),
}
print(json.dumps(summary, indent=2))
print("\nFirst chunks:")
for chunk_path in chunk_paths[:5]:
    print(" -", chunk_path)
